# 01 — Get the data

**Goal.** Obtain the raw data and check that it is the right version.

The raw CSV files are **not stored in this repository**: they are too large for GitHub
(`appearances.csv` alone is about 200 MB, above GitHub's 100 MB limit per file).
Anyone who clones the repository therefore starts with an empty `data/raw/` folder.
This notebook explains where the data comes from, downloads it if it is missing, and checks it.

**Output:** the five CSV files used by the project in `data/raw/`, and a summary table in `tables/raw_data_summary.csv`.

## 1. Source, licence and version

| | |
|---|---|
| **Dataset** | *Football Data from Transfermarkt* |
| **Author** | David Cariboo (`dcaribou/transfermarkt-datasets` on GitHub) |
| **Where** | Kaggle: https://www.kaggle.com/datasets/davidcariboo/player-scores |
| **How it was built** | Scraped from the Transfermarkt website, then cleaned with Python and SQL |
| **Licence** | The GitHub project is released under CC0 1.0 (public domain). *[Check the licence shown on the Kaggle page and confirm it here.]* |
| **Version used** | Frozen dataset: automatic updates stopped in July 2026. Latest match in `appearances`: 28 June 2026; latest valuation: 12 June 2026. |

Because the dataset is no longer updated, every download gives the same files, which makes the results reproducible.
Section 4 checks this by comparing row counts with the version used in this project.

**Files used by the project:**

| File | Content |
|---|---|
| `appearances.csv` | One row per player per match: minutes, goals, assists, cards |
| `player_valuations.csv` | One row per market-value update of a player (defines the target) |
| `players.csv` | Player profiles: birth date, position |
| `clubs.csv` | Club information |
| `competitions.csv` | Leagues and cups |
| `game_lineup.csv` | Team Composition  |
| `Transfers.csv` | Changes of team by the player  |

The other files of the dataset (`game_lineups`, `transfers`, `games`, ...) are not needed for the main analysis.

## 2. Setup

Import the project paths from `src/config.py` and create `data/raw/` if it does not exist yet.

In [ ]:
import sys
sys.path.append("..")                                 # allows importing from src/
from src import config
import pandas as pd

config.DATA_RAW.mkdir(parents=True, exist_ok=True)   # create data/raw/ if needed
config.TABLES.mkdir(parents=True, exist_ok=True)     # create tables/ if needed

# Files of the project, with the number of data rows in the version used here.
# None = optional file whose reference row count has not been recorded yet:
# run section 4 once and copy the counts here.
EXPECTED_ROWS = {
    "appearances.csv": 1_894_350,
    "player_valuations.csv": 656_301,
    "players.csv": 50_149,
    "clubs.csv": 796,
    "competitions.csv": 65,
    "game_lineups.csv": None,      # optional
    "transfers.csv": None,         # optional
}

## 3. Download the data (only if files are missing)



**Option A  — automatic download with the Kaggle API.**
Requires `pip install kaggle` and a Kaggle API token saved in `~/.kaggle/kaggle.json`

**Option B— manual download (simplest).**
1. Open https://www.kaggle.com/datasets/davidcariboo/player-scores (a free Kaggle account is needed).
2. Click **Download**, unzip the archive.
3. Copy the seven files listed above into `data/raw/`.

The cell below does nothing if all files are already present. Otherwise it tries option A,
and prints the instructions of option B if the Kaggle API is not available.

In [ ]:
# Which project files are missing from data/raw/?
missing = [f for f in EXPECTED_ROWS if not (config.DATA_RAW / f).exists()]

if not missing:
    print("All files are already in data/raw/: nothing to download.")
else:
    print("Missing files:", missing)
    try:
        import kaggle                                  # needs a token in ~/.kaggle/kaggle.json
        kaggle.api.authenticate()
        kaggle.api.dataset_download_files("davidcariboo/player-scores",
                                          path=str(config.DATA_RAW), unzip=True)
        print("Download finished.")
    except Exception as error:
        print("Automatic download not possible:", error)
        print("Please follow option A above (manual download from Kaggle).")

## 4. Check the files

For each file we check that it exists, measure its size, and count its rows.
The counts are compared with the version used in this project.

**What we expect to see:** every main file marked `OK`. A `DIFFERENT` status means another version
of the dataset was downloaded, and results may differ slightly from the paper.

Rows are counted by reading the file line by line, which is faster and lighter than loading it with pandas.

In [ ]:
def count_rows(path):
    """Number of data rows in a CSV file (all lines minus the header)."""
    with open(path, encoding="utf-8") as f:
        return sum(1 for _ in f) - 1

summary = []
for name, expected in EXPECTED_ROWS.items():
    path = config.DATA_RAW / name
    if not path.exists():
        summary.append({"file": name, "size_mb": None, "rows": None,
                        "expected_rows": expected, "status": "MISSING"})
        continue
    rows = count_rows(path)
    summary.append({"file": name,
                    "size_mb": round(path.stat().st_size / 1e6, 1),
                    "rows": rows,
                    "expected_rows": expected,
                    "status": ("NO REFERENCE" if expected is None
                               else "OK" if rows == expected else "DIFFERENT")})

summary = pd.DataFrame(summary)
print(summary)